# Doblaje Expresivo en Español con Chatterbox Multilingual (GPU T4 Gratis)

Genera voces con **pasión, rabia, urgencia y desgarro** usando [Chatterbox Multilingual](https://github.com/resemble-ai/chatterbox) + cadena de procesado `ffmpeg` para combate/drama, conectado directamente a tu carpeta `Creador-de-mundos-Rpg-Colab` en Google Drive.

### Pasos
1. Activa GPU en Colab: **`Entorno de ejecución` → `Cambiar tipo de entorno de ejecución` → `GPU T4`**.
2. Ejecuta las celdas en orden (ya no hace falta subir JSON ni scripts a mano).
3. *(Opcional)* Usa la celda 3A o 3B para subir o grabar con tu micrófono 5–10 s de referencia emocional en `referencias_voz/`.

In [ ]:
# 1. Verificar GPU T4 e instalar Chatterbox Multilingual sin romper PyTorch/NumPy de Colab
!nvidia-smi
!apt-get -qq update && apt-get -qq install -y ffmpeg
!pip install -q --no-deps chatterbox-tts
!pip install -q s3tokenizer conformer==0.3.2 diffusers==0.29.0 librosa==0.11.0 "safetensors>=0.8.0" omegaconf pyloudnorm pykakasi==2.3.0 "resemble-perth @ git+https://github.com/resemble-ai/Perth.git@master"

In [ ]:
# 2. Conectar automáticamente con Google Drive (Creador-de-mundos-Rpg-Colab)
from google.colab import drive, files
import os, shutil

drive.mount('/content/drive')
WORK_DIR = '/content/drive/MyDrive/Creador-de-mundos-Rpg-Colab'
if os.path.isdir(WORK_DIR):
    os.chdir(WORK_DIR)
    print('Carpeta de Drive conectada:', os.getcwd())
else:
    print('No se encontró', WORK_DIR, '— creando carpeta local y pidiendo subida manual...')
    os.makedirs('tools', exist_ok=True)
    uploaded = files.upload()
    if os.path.exists('generar-voces-chatterbox.py') and not os.path.exists('tools/generar-voces-chatterbox.py'):
        shutil.move('generar-voces-chatterbox.py', 'tools/generar-voces-chatterbox.py')

os.makedirs('referencias_voz', exist_ok=True)
assert os.path.exists('tools/generar-voces-chatterbox.py'), 'Falta tools/generar-voces-chatterbox.py'
assert os.path.exists('voces_skills.json') or os.path.exists('dialogos_voces.json'), 'Falta voces_skills.json o dialogos_voces.json'
!python tools/generar-voces-chatterbox.py --listar-perfiles | head -n 15

In [ ]:
# 3A. (OPCIONAL) Subir muestras de voz de referencia (5-10s .wav/.mp3/.ogg) a Drive
# Nombres reconocidos: heroe_guerrero.wav, heroe_mago.wav, brenda.wav, herrero_brann.wav, narrador.wav,
# o por grupo: combate_hombre.wav, combate_mujer.wav, dramatico_hombre.wav, dramatico_mujer.wav, default.wav
from google.colab import files
import shutil, os

refs = files.upload()
for nombre in refs:
    shutil.move(nombre, os.path.join('referencias_voz', nombre))
print('Referencias listas en Drive:', os.listdir('referencias_voz'))

In [ ]:
# 3B. (OPCIONAL) Grabar 6 segundos directamente con tu micrófono en Colab (se guarda en Drive)
# Cambia NOMBRE_REF (ej: 'combate_hombre', 'herrero_brann', 'combate_mujer') y ejecuta la celda.
NOMBRE_REF = 'combate_hombre'
SEGUNDOS = 6

from IPython.display import Javascript, display, Audio
from google.colab import output
import base64, os

RECORD_JS = f"""
const sleep = time => new Promise(r => setTimeout(r, time));
const record = () => new Promise(async resolve => {{
  const stream = await navigator.mediaDevices.getUserMedia({{ audio: true }});
  const rec = new MediaRecorder(stream);
  const chunks = [];
  rec.ondataavailable = e => chunks.push(e.data);
  rec.start();
  await sleep({SEGUNDOS * 1000});
  rec.onstop = async () => {{
    stream.getTracks().forEach(t => t.stop());
    const blob = new Blob(chunks);
    const reader = new FileReader();
    reader.readAsDataURL(blob);
    reader.onloadend = () => resolve(reader.result.split(',')[1]);
  }};
  rec.stop();
}});
record();
"""
print(f'Habla o grita con garra durante {SEGUNDOS}s...')
b64 = output.eval_js(RECORD_JS)
webm_path = f'referencias_voz/{NOMBRE_REF}.webm'
wav_path = f'referencias_voz/{NOMBRE_REF}.wav'
with open(webm_path, 'wb') as f:
    f.write(base64.b64decode(b64))
!ffmpeg -y -loglevel error -i {webm_path} -ac 1 -ar 24000 {wav_path}
os.remove(webm_path)
display(Audio(wav_path))

In [ ]:
# 4. Prueba rápida de 3 gritos de combate para escuchar la pasión antes de generar todo
EXAGGERATION_MULT = 0.90

!python tools/generar-voces-chatterbox.py --json voces_skills.json --solo-id sk_sismo sk_nova sk_bw_bolt comp_brenda_ataque --forzar --exaggeration-mult {EXAGGERATION_MULT}

from IPython.display import Audio, display
for f in ['assets/audio/voces/skills/sk_sismo.ogg', 'assets/audio/voces/skills/sk_nova.ogg', 'assets/audio/voces/skills/sk_bw_bolt.ogg', 'assets/audio/voces/skills/comp_brenda_ataque.ogg']:
    print(f)
    display(Audio(f))

In [ ]:
# 5. Generar TODAS las 50 voces de skills (y opcionalmente los diálogos clave)
!python tools/generar-voces-chatterbox.py --json voces_skills.json --forzar --exaggeration-mult {EXAGGERATION_MULT}

# Si también quieres regenerar personajes clave de la historia (o todos), descomenta:
# !python tools/generar-voces-chatterbox.py --json dialogos_voces.json --solo-personaje herrero_brann --forzar
# !python tools/generar-voces-chatterbox.py --json dialogos_voces.json --solo-personaje narrador --forzar
# !python tools/generar-voces-chatterbox.py --json dialogos_voces.json --forzar

In [ ]:
# 6. Guardar voces_chatterbox_ogg.zip en Drive (y ofrecer descarga al navegador)
import os, shutil
from google.colab import files

zip_path = shutil.make_archive('voces_chatterbox_ogg', 'zip', '.', 'assets/audio/voces')
print('ZIP guardado en Drive:', os.path.abspath(zip_path))
try:
    files.download('voces_chatterbox_ogg.zip')
except Exception as e:
    print('Descarga directa omitida (ya quedó guardado en tu Google Drive):', e)